# rlpt_0f — Le SOTA de l'alignement en ligne : `trl.GRPOTrainer` contre une boucle PPO maison

**Série** Post-Training RL (`rlpt_*`) — niveau intermédiaire · **Prérequis** : [`rlpt_0`](rlpt_0_reward_model_from_scratch.ipynb) (le monde synthétique et le reward model Bradley-Terry, repris verbatim), [`rlpt_1`](rlpt_1_ppo_lm_rlhf.html) (PPO from scratch), [`rlpt_0e`](rlpt_0e_trl_DPO_SOTA.ipynb) (le geste SOTA côté offline, même monde).

Ce notebook ferme l'arc ouvert par [`rlpt_0e`](rlpt_0e_trl_DPO_SOTA.ipynb) : là où DPO apprenait
**offline** (un dataset figé de préférences), ici la politique **échantillonne ses propres
réponses** et apprend en ligne du score d'un juge — l'arc complet RLHF de `rlpt_1`.
Le geste SOTA est le même : brancher la lib industrielle sur notre monde et départager
à budget égal.

Une découverte d'abord, mesurée dans la cellule d'environnement : **`trl.PPOTrainer`
n'existe plus**. L'issue #16063 (bloc B.6) demandait « `trl.PPOTrainer` sur la même
mini-tâche que rlpt_1 » — la lib a supprimé ce trainer en cours de route, et son pendent
industriel actuel est `trl.GRPOTrainer`. Ce notebook documente la succession, puis
compare :

| Bras | Algorithme | Baseline de l'avantage | Pile |
|---|---|---|---|
| **Maison** | PPO-clip (recette `rlpt_1` transplantée) | value net **appris** $V(x)$ | torch pur |
| **SOTA** | GRPO (group relative policy optimization) | **moyenne du groupe** de $G$ réponses | `trl` |

Les deux bras résolvent le même problème — *réduire la variance d'une récompense terminale
unique* — par deux baselines différentes. C'est LA différence pédagogique du notebook :
un critic entraîné, ou la structure du groupe elle-même.

In [1]:
import sys, time, tempfile, platform
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import transformers, trl, datasets
from scipy.stats import spearmanr

print(f"python {sys.version.split()[0]} | torch {torch.__version__} | transformers {transformers.__version__}"
      f" | trl {trl.__version__} | datasets {datasets.__version__}")

# Le fait structurant : PPOTrainer a disparu de trl
exports = [x for x in dir(trl) if 'Trainer' in x]
print("trainers trl :", ', '.join(sorted(exports)))
print("PPOTrainer present :", hasattr(trl, 'PPOTrainer'))
assert not hasattr(trl, 'PPOTrainer'), "si cette assert casse, trl a resuscite PPOTrainer -- relire le notebook" 

python 3.11.9 | torch 2.13.0+cpu | transformers 5.12.1 | trl 1.10.0 | datasets 5.0.0
trainers trl : DPOTrainer, DistillationTrainer, GRPOTrainer, KTOTrainer, RLOOTrainer, RewardTrainer, SFTTrainer
PPOTrainer present : False


`trl.PPOTrainer` a été retiré de la lib (deprecated fin 2024, supprimé depuis) : les
traineurs **en ligne** que trl expose aujourd'hui sont `GRPOTrainer` et `RLOOTrainer`.
Le « PPO-RLHF SOTA comparison » de l'issue se livre donc contre `GRPOTrainer` — le
successeur factuel, celui que la pratique industrielle a adopté (c'est le cœur de
`rlpt_2` côté main, et des notebooks `PT-11a/b` côté échelle réelle).

**GRPO en une phrase** : au lieu d'entraîner un critic $V(s)$ à prédire la récompense,
on génère $G$ réponses au même prompt, et l'avantage de chacune est
$A_i = (r_i - \text{mean}(r_{1..G})) / \text{std}(r_{1..G})$ — le groupe est sa propre
baseline. Pas de value net, pas de GAE : la variance se réduit par la structure même
de l'échantillonnage.

## 1. Le monde synthétique (verbatim `rlpt_0`)

Même monde que `rlpt_0`/`rlpt_0e` : vocabulaire de 8 lettres `a..h` à poids connus,
deux prompts `<pA>`/`<pB>` avec bonus positionnels conditionnels, réponses de 8 tokens.
La **vraie récompense** $r^*$ sert uniquement de **métrique** — l'entraînement, lui,
ne verra jamais que le score du juge appris (section 2).

In [2]:
TOK = ['<pA>', '<pB>', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', '<eos>']
V = {c: i for i, c in enumerate(TOK)}
LEN_R = 8
W_TOK = {'a': 0.30, 'b': 0.05, 'c': 0.10, 'd': 0.15, 'e': 0.25, 'f': -0.10, 'g': -0.15, 'h': -0.20}
SEEDS = [0, 1, 7, 42]

def true_reward(seq, prompt):
    r = sum(W_TOK[TOK[t]] for t in seq)
    if prompt == 0 and seq[0] == V['a']:
        r += 0.8
    if prompt == 1 and seq[4] == V['e']:
        r += 0.8
    return r

def sample_response(rng):
    return rng.integers(2, len(TOK) - 1, size=LEN_R)   # contenu seulement

def make_pairs(n_pairs, rng, min_gap=0.3, beta=1.0):
    pw, y, pr = [], [], []
    while len(pw) < n_pairs:
        prompt = int(rng.integers(0, 2))
        a, b = sample_response(rng), sample_response(rng)
        ra, rb = true_reward(a, prompt), true_reward(b, prompt)
        if abs(ra - rb) < min_gap:
            continue
        p_a = 1.0 / (1.0 + np.exp(-beta * (ra - rb)))
        pw.append((a, b)); y.append(1 if rng.random() < p_a else 0); pr.append(prompt)
    return pw, np.array(y), np.array(pr)

rng0 = np.random.default_rng(7)
fond = np.mean([true_reward(sample_response(rng0), int(rng0.integers(0, 2))) for _ in range(4000)])
print(f"bruit de fond (politique uniforme sur le contenu) : {fond:.3f}")
print(f"plafond glouton : <pA> -> a*8 + bonus = {8*0.30+0.8:.2f} | <pB> -> a*8 = {8*0.30:.2f}")

bruit de fond (politique uniforme sur le contenu) : 0.487
plafond glouton : <pA> -> a*8 + bonus = 3.20 | <pB> -> a*8 = 2.40


Deux repères pour lire toutes les courbes de ce notebook : une politique uniforme sur
les lettres marque ~0,44 en moyenne (le bonus positionnel tombe rarement), et la
meilleure réponse possible vaut 3,2 (`<pA>`) / 2,4 (`<pB>`). Un policy à 0,8-0,9 a
donc déjà appris beaucoup — sans pour autant épuiser le monde.

## 2. Le juge : reward model Bradley-Terry (recette `rlpt_0`, compressée)

L'arc RLHF complet exige un **reward model appris** — pas la vraie récompense. On
réentraîne le RM de `rlpt_0` : paires de préférences bruitées par un Bradley-Terry,
MLE par paires, ~5 s. **Les deux bras (PPO maison et GRPO) optimisent le score de CE
RM** — le même juge, les mêmes données. La vraie récompense n'intervient qu'aux
évaluations : c'est là que se mesurera l'écart juge/vrai (la faille de tout RLHF).

In [3]:
def encode(pair_list, prompt_list):
    B = len(pair_list)
    x = torch.zeros(B, 2, 1 + LEN_R, dtype=torch.long)
    for i, (a, b) in enumerate(pair_list):
        x[i, 0, 0] = x[i, 1, 0] = prompt_list[i]
        x[i, 0, 1:] = torch.from_numpy(np.asarray(a, dtype=np.int64))
        x[i, 1, 1:] = torch.from_numpy(np.asarray(b, dtype=np.int64))
    return x

class RewardModel(nn.Module):
    def __init__(self, vs=len(TOK), hid=64):
        super().__init__()
        self.emb = nn.Embedding(vs, hid)
        self.pos = nn.Embedding(1 + LEN_R, hid)
        self.mlp = nn.Sequential(nn.Linear(hid, hid), nn.ReLU(), nn.Linear(hid, 1))
    def forward(self, x):
        L = x.shape[1]
        h = self.emb(x) + self.pos(torch.arange(L))
        return self.mlp(h.mean(1)).squeeze(-1)

def train_bt(model, x, y, epochs=60, bs=256, lr=1e-2):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    n = len(y)
    for ep in range(epochs):
        perm = torch.randperm(n).tolist()
        for i in range(0, n, bs):
            idx = perm[i:i + bs]
            xb, yb = x[idx], torch.tensor(y[idx], dtype=torch.long)
            b = len(idx)
            win = xb[torch.arange(b), 1 - yb]
            lose = xb[torch.arange(b), yb]
            loss = F.binary_cross_entropy_with_logits(model(win) - model(lose), torch.ones(b))
            opt.zero_grad(); loss.backward(); opt.step()

def make_rm(seed):
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    pw, y, pr = make_pairs(4000, rng)
    rm = RewardModel()
    train_bt(rm, encode(pw, pr), y)
    return rm

t0 = time.time()
RM = make_rm(0)
print(f"RM (seed 0) entraîné en {time.time()-t0:.1f}s -- c'est le juge des deux bras")

RM (seed 0) entraîné en 5.3s -- c'est le juge des deux bras


## 3. La politique, la métrique, et le garde-fou d'évaluation

Le LM est celui de `rlpt_0e` : GPT-2 de jouet (vocab 11, 2 couches, 64 dim, ~100k
paramètres). La **métrique** est commune aux deux bras : on échantillonne la politique
(sans masque — elle peut émettre n'importe quoi), on filtre les tokens de contenu
(`a..h`), on complète avec `h` (le pire poids) si elle bavarde du prompt/EOS — la
dégénérescence est punie, pas récompensée — et on score à la **vraie** récompense.

In [4]:
from transformers import AutoModelForCausalLM, GPT2Config, PreTrainedTokenizerFast
from tokenizers import Tokenizer, models, pre_tokenizers

def make_tz():
    tm = models.WordLevel(vocab={c: i for i, c in enumerate(TOK)}, unk_token='<eos>')
    tk = Tokenizer(tm)
    tk.pre_tokenizer = pre_tokenizers.WhitespaceSplit()   # pas Whitespace (regex qui déchire <pB>)
    return PreTrainedTokenizerFast(tokenizer_object=tk, pad_token='<pA>', bos_token='<pA>', eos_token='<eos>')

tz = make_tz()
assert tz('<pB> a b c d e f g h')["input_ids"] == [1, 2, 3, 4, 5, 6, 7, 8, 9]

def make_lm(seed):
    torch.manual_seed(seed)
    cfg = GPT2Config(vocab_size=len(TOK), n_positions=16, n_embd=64, n_layer=2, n_head=4,
                     bos_token_id=0, eos_token_id=V['<eos>'], pad_token_id=0)
    return AutoModelForCausalLM.from_config(cfg)

def eval_policy(model, n=384, seed=123):
    """Vraie récompense moyenne de la politique, échantillonnée sans masque."""
    g = np.random.default_rng(seed)
    tot = 0.0
    for _ in range(n):
        pid = int(g.integers(0, 2))
        ids = [pid]
        for _ in range(LEN_R + 2):
            with torch.no_grad():
                logits = model(torch.tensor([ids])).logits[0, -1]
            ids.append(int(torch.multinomial(F.softmax(logits, -1), 1)))
        content = [t for t in ids if 2 <= t <= 9][:LEN_R]
        content = content + [V['h']] * (LEN_R - len(content))
        tot += true_reward(np.array(content, dtype=np.int64), pid)
    return tot / n

torch.manual_seed(0)
pol_test = make_lm(0)
print(f"politique non entraînée : vraie récompense {eval_policy(pol_test):.3f}"
      f"  (le padding 'h' punit les bavardages prompt/EOS)")

[transformers] We strongly recommend passing in an `attention_mask` since your input_ids may be padded. See https://huggingface.co/docs/transformers/troubleshooting#incorrect-output-when-padding-tokens-arent-masked.
You may ignore this warning if your `pad_token_id` (0) is identical to the `bos_token_id` (0), `eos_token_id` (10), or the `sep_token_id` (None), and your input is not padded.


politique non entraînée : vraie récompense 0.344  (le padding 'h' punit les bavardages prompt/EOS)


## 4. Bras maison : PPO-clip avec value net (recette `rlpt_1` transplantée)

La boucle de `rlpt_1`, adaptée au monde TOK : rollouts masqués au contenu, récompense
terminale = score RM, **avantage = $r - V(x)$** où $V$ est un value net appris (même
architectecture que le RM), ratio clip PPO sur les log-vraisemblances de tokens, et
pénalité KL $\beta$ contre la politique de référence (approximation $k_3$, la même
que dans `trl`) — le tout en ~50 lignes.

In [5]:
class ValueNet(nn.Module):
    def __init__(self, vs=len(TOK), hid=64):
        super().__init__()
        self.emb = nn.Embedding(vs, hid)
        self.pos = nn.Embedding(1 + LEN_R, hid)
        self.mlp = nn.Sequential(nn.Linear(hid, hid), nn.ReLU(), nn.Linear(hid, 1))
    def forward(self, x):
        L = x.shape[1]
        h = self.emb(x) + self.pos(torch.arange(L))
        return self.mlp(h.mean(1)).squeeze(-1)

def token_logps(model, x):
    out = torch.log_softmax(model(x).logits, dim=-1)
    tgt = x[:, 1:]
    return out[:, :-1, :].gather(-1, tgt.unsqueeze(-1)).squeeze(-1)   # (B, L-1)

def rollout(policy, n, gen):
    """n trajectoires : prompt + 8 tokens échantillonnés, masqués au contenu."""
    seqs, pids = [], []
    for _ in range(n):
        pid = int(gen.integers(0, 2))
        ids = [pid]
        for _ in range(LEN_R):
            with torch.no_grad():
                logits = policy(torch.tensor([ids])).logits[0, -1]
            mask = torch.full((len(TOK),), -1e9)
            mask[2:10] = 0.0
            ids.append(int(torch.multinomial(F.softmax(logits + mask, -1), 1)))
        seqs.append(ids[1:]); pids.append(pid)
    return seqs, pids

def seqs_to_x(seqs, pids):
    B = len(seqs)
    x = torch.zeros(B, 1 + LEN_R, dtype=torch.long)
    for i, (s, p) in enumerate(zip(seqs, pids)):
        x[i, 0] = p
        x[i, 1:] = torch.from_numpy(np.asarray(s, dtype=np.int64))
    return x

BETA_KL, PPO_STEPS, PPO_BS = 0.05, 12, 192    # 12 x 192 = 2304 rollouts

def train_ppo_maison(policy, ref, value, rm, seed,
                     steps=PPO_STEPS, bs=PPO_BS, lr=3e-4, beta=BETA_KL, clip=0.2, vf=0.5):
    opt = torch.optim.Adam(list(policy.parameters()) + list(value.parameters()), lr=lr)
    gen = np.random.default_rng(seed)
    hist = []
    for it in range(steps):
        seqs, pids = rollout(policy, bs, gen)
        x = seqs_to_x(seqs, pids)
        with torch.no_grad():
            r = rm(x)
            v = value(x)
            adv = (r - v)
            adv = (adv - adv.mean()) / (adv.std() + 1e-8)
            old_lp = token_logps(policy, x)
            ref_lp = token_logps(ref, x)
        lp = token_logps(policy, x)
        ratio = torch.exp((lp - old_lp).sum(1))
        pg = torch.minimum(ratio * adv, torch.clamp(ratio, 1 - clip, 1 + clip) * adv).mean()
        kl = (torch.exp(ref_lp - lp) - (ref_lp - lp) - 1).sum(1).mean()
        vloss = F.mse_loss(value(x), r)
        loss = -pg + vf * vloss + beta * kl
        opt.zero_grad(); loss.backward(); opt.step()
        hist.append(dict(r=float(r.detach().mean()), kl=float(kl.detach()),
                         vloss=float(vloss.detach())))
    return hist

In [6]:
t0 = time.time()
torch.manual_seed(0)
pol_m, ref_m = make_lm(0), make_lm(0)
value_m = ValueNet()
r_avant = eval_policy(pol_m)
hist_m = train_ppo_maison(pol_m, ref_m, value_m, RM, seed=0)
t_maison = time.time() - t0
r_apres = eval_policy(pol_m)
print(f"[Bras maison | PPO clip + value net]  {t_maison:.1f}s")
print(f"vraie récompense : {r_avant:.3f} -> {r_apres:.3f}  (bruit de fond {fond:.3f})")
for h in hist_m[::4]:
    print(f"  it r_rm={h['r']:+.3f} kl={h['kl']:.3f} vloss={h['vloss']:.3f}")

[Bras maison | PPO clip + value net]  68.0s
vraie récompense : 0.219 -> 0.815  (bruit de fond 0.487)
  it r_rm=-0.184 kl=0.042 vloss=0.826
  it r_rm=+0.606 kl=0.462 vloss=1.216
  it r_rm=+0.769 kl=0.863 vloss=1.526


**Lecture du bras maison.** Le value net met quelques itérations à caler (la `vloss`
monte d'abord : $V$ apprend pendant que la politique bouge), puis l'avantage
$r - V$ devient informative et la récompense RM décolle. C'est **le** coût caché
du PPO : un critic à entraîner en même temps que la politique, sur un signal
non stationnaire (la cible de $V$ change à chaque update). À l'échelle réelle,
ce critic coûte un modèle entier ; ici 5 505 paramètres suffisent.

## 5. Bras SOTA : `trl.GRPOTrainer`

Trois différences d'ingénierie avec le bras maison, toutes instructives :

1. **Le reward est une fonction** — `reward_funcs` reçoit `(prompts, completions,
   completion_ids)` en **arguments nommés** et rend une liste de floats. Le juge
   peut être n'importe quoi : ici notre RM, ailleurs un vérificateur Z3, un
   classifieur, une heuristique. C'est l'interface qui a rendu GRPO populaire
   (RLVR — rewards vérifiables).
2. **La baseline est dans la structure** : `num_generations=8` = 8 réponses par
   prompt, avantage normalisé dans le groupe. Pas de value net.
3. **`beta > 0` exige un modèle de référence identifiable** : avec un LM construit
   `from_config` (sans path), trl essaie de recharger un ref depuis le hub et
   échoue (`Repo id must use...`). Le workaround propre : `save_pretrained` vers
   un dossier temporaire et passer le **chemin**.

In [7]:
from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer

N_PROMPTS, G = 96, 8          # 96 x 8 x 3 époques = 2304 completions (budget PPO)

def reward_rm(prompts, completions, completion_ids=None, **kw):
    """Score du juge RM sur les complétions -- même filtre contenu que eval_policy."""
    out = []
    for p, ids in zip(prompts, completion_ids):
        pid = V[p.strip()]
        content = [t for t in ids if 2 <= t <= 9][:LEN_R]
        content = content + [V['h']] * (LEN_R - len(content))
        x = torch.zeros(1, 1 + LEN_R, dtype=torch.long)
        x[0, 0] = pid
        x[0, 1:] = torch.from_numpy(np.asarray(content, dtype=np.int64))
        with torch.no_grad():
            out.append(float(RM_SEED_LOCAL(x)[0]))
    return out

def train_grpo_trl(seed, n_epochs=3):
    pol = make_lm(seed)
    pol_dir = tempfile.mkdtemp(prefix='rlpt0f_pol_')
    pol.save_pretrained(pol_dir)          # workaround : beta>0 veut un ref par chemin
    rng = np.random.default_rng(seed)
    ds = Dataset.from_list(
        [{"prompt": TOK[int(rng.integers(0, 2))] + ' '} for _ in range(N_PROMPTS)])
    cfg = GRPOConfig(
        output_dir=tempfile.mkdtemp(prefix='rlpt0e_grpo_'),   # hors du dépôt
        beta=BETA_KL,                      # même coefficient KL que le bras maison
        num_generations=G,
        max_completion_length=LEN_R + 1,
        per_device_train_batch_size=32,
        num_train_epochs=n_epochs,
        learning_rate=1e-4,
        logging_steps=8,
        report_to=[], save_strategy='no', disable_tqdm=True,
        seed=seed, use_cpu=True,
    )
    tr = GRPOTrainer(model=pol_dir, reward_funcs=reward_rm, args=cfg,
                     train_dataset=ds, processing_class=tz)
    tr.train()
    return tr

Budget apparié : le bras maison fait 12 itérations × 192 rollouts = **2 304
réponses évaluées** ; le bras GRPO fait 96 prompts × 8 générations × 3 époques =
**2 304 réponses**. Même juge, même budget d'échantillons, même $\beta$ — les
hyperparamètres restants (lr, clip) sont ceux de chaque méthode par défaut au jouet.

In [8]:
RM_SEED_LOCAL = RM     # le juge global sert au run seed 0 ; run_seed repose le sien

t0 = time.time()
torch.manual_seed(0)
tr0 = train_grpo_trl(0)
t_trl = time.time() - t0
r_trl = eval_policy(tr0.model)
print(f"[Bras SOTA | trl.GRPOTrainer]  {t_trl:.1f}s")
print(f"vraie récompense : {r_trl:.3f}  (maison {r_apres:.3f}, bruit de fond {fond:.3f})")
last = tr0.state.log_history[-2]
print(f"log trl : reward={last.get('reward', float('nan')):.3f} kl={last.get('kl', float('nan')):.3f}"
      f" frac_zero_std={last.get('frac_reward_zero_std', float('nan')):.2f}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/28 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/28 [00:00<?, ?it/s]

{'loss': '-0.3013', 'grad_norm': '1.19', 'learning_rate': '9.028e-05', 'num_tokens': '1990', 'completions/mean_length': '6.773', 'completions/min_length': '1', 'completions/max_length': '9', 'completions/clipped_ratio': '0.4375', 'completions/mean_terminated_length': '5.061', 'completions/min_terminated_length': '1', 'completions/max_terminated_length': '9', 'rewards/reward_rm/mean': '-1.828', 'rewards/reward_rm/std': '2.583', 'reward': '-1.828', 'reward_std': '2.583', 'frac_reward_zero_std': '0', 'kl': '0.008674', 'entropy': '2.369', 'clip_ratio/low_mean': '0', 'clip_ratio/high_mean': '0', 'clip_ratio/region_mean': '0', 'clip_ratio/low_min': '0', 'clip_ratio/high_max': '0', 'step_time': '0.395', 'epoch': '0.3333'}


{'loss': '-0.2663', 'grad_norm': '0.8682', 'learning_rate': '7.917e-05', 'num_tokens': '4079', 'completions/mean_length': '7.16', 'completions/min_length': '1.125', 'completions/max_length': '9', 'completions/clipped_ratio': '0.5', 'completions/mean_terminated_length': '5.343', 'completions/min_terminated_length': '1.125', 'completions/max_terminated_length': '9', 'rewards/reward_rm/mean': '-1.225', 'rewards/reward_rm/std': '2.281', 'reward': '-1.225', 'reward_std': '2.281', 'frac_reward_zero_std': '0', 'kl': '0.02859', 'entropy': '2.358', 'clip_ratio/low_mean': '0', 'clip_ratio/high_mean': '0', 'clip_ratio/region_mean': '0', 'clip_ratio/low_min': '0', 'clip_ratio/high_max': '0', 'step_time': '0.3985', 'epoch': '0.6667'}


...

{'loss': '-0.2078', 'grad_norm': '0.9393', 'learning_rate': '3.472e-05', 'num_tokens': '1.305e+04', 'completions/mean_length': '7.691', 'completions/min_length': '1.25', 'completions/max_length': '9', 'completions/clipped_ratio': '0.6172', 'completions/mean_terminated_length': '5.497', 'completions/min_terminated_length': '1.25', 'completions/max_terminated_length': '9', 'rewards/reward_rm/mean': '-0.2605', 'rewards/reward_rm/std': '2.036', 'reward': '-0.2605', 'reward_std': '2.036', 'frac_reward_zero_std': '0', 'kl': '0.08299', 'entropy': '2.322', 'clip_ratio/low_mean': '0', 'clip_ratio/high_mean': '0', 'clip_ratio/region_mean': '0', 'clip_ratio/low_min': '0', 'clip_ratio/high_max': '0', 'step_time': '0.3585', 'epoch': '2'}


{'loss': '-0.1951', 'grad_norm': '0.7103', 'learning_rate': '2.361e-05', 'num_tokens': '1.528e+04', 'completions/mean_length': '7.703', 'completions/min_length': '1.5', 'completions/max_length': '9', 'completions/clipped_ratio': '0.6445', 'completions/mean_terminated_length': '5.33', 'completions/min_terminated_length': '1.5', 'completions/max_terminated_length': '9', 'rewards/reward_rm/mean': '-0.4443', 'rewards/reward_rm/std': '2.101', 'reward': '-0.4443', 'reward_std': '2.101', 'frac_reward_zero_std': '0', 'kl': '0.08399', 'entropy': '2.317', 'clip_ratio/low_mean': '0', 'clip_ratio/high_mean': '0', 'clip_ratio/region_mean': '0', 'clip_ratio/low_min': '0', 'clip_ratio/high_max': '0', 'step_time': '0.3537', 'epoch': '2.333'}


{'loss': '-0.1918', 'grad_norm': '0.6812', 'learning_rate': '1.25e-05', 'num_tokens': '1.753e+04', 'completions/mean_length': '7.809', 'completions/min_length': '1.125', 'completions/max_length': '9', 'completions/clipped_ratio': '0.6797', 'completions/mean_terminated_length': '5.298', 'completions/min_terminated_length': '1.125', 'completions/max_terminated_length': '9', 'rewards/reward_rm/mean': '-0.4361', 'rewards/reward_rm/std': '2.122', 'reward': '-0.4361', 'reward_std': '2.122', 'frac_reward_zero_std': '0', 'kl': '0.08295', 'entropy': '2.313', 'clip_ratio/low_mean': '0', 'clip_ratio/high_mean': '0', 'clip_ratio/region_mean': '0', 'clip_ratio/low_min': '0', 'clip_ratio/high_max': '0', 'step_time': '0.35', 'epoch': '2.667'}


{'loss': '-0.1609', 'grad_norm': '0.6709', 'learning_rate': '1.389e-06', 'num_tokens': '1.984e+04', 'completions/mean_length': '8.004', 'completions/min_length': '1.625', 'completions/max_length': '9', 'completions/clipped_ratio': '0.6953', 'completions/mean_terminated_length': '5.711', 'completions/min_terminated_length': '1.625', 'completions/max_terminated_length': '9', 'rewards/reward_rm/mean': '-0.2654', 'rewards/reward_rm/std': '1.852', 'reward': '-0.2654', 'reward_std': '1.852', 'frac_reward_zero_std': '0', 'kl': '0.08312', 'entropy': '2.311', 'clip_ratio/low_mean': '0', 'clip_ratio/high_mean': '0', 'clip_ratio/region_mean': '0', 'clip_ratio/low_min': '0', 'clip_ratio/high_max': '0', 'step_time': '0.357', 'epoch': '3'}
{'train_runtime': '27.23', 'train_samples_per_second': '10.57', 'train_steps_per_second': '2.644', 'train_loss': '-0.2145', 'epoch': '3'}


[Bras SOTA | trl.GRPOTrainer]  27.6s
vraie récompense : 0.880  (maison 0.815, bruit de fond 0.487)
log trl : reward=-0.265 kl=0.083 frac_zero_std=0.00


## 6. Multi-seed {0, 1, 7, 42}

Chaque seed **retire tout le pipeline** : nouvelles paires, nouveau RM, nouvelles
politiques. C'est la force du monde synthétique — la variabilité du juge lui-même
entre dans la mesure.

In [9]:
def run_seed(seed):
    global RM_SEED_LOCAL
    torch.manual_seed(seed)
    RM_SEED_LOCAL = make_rm(seed)
    out = {}

    t0 = time.time()
    pol, ref = make_lm(seed), make_lm(seed)
    value = ValueNet()
    r0 = eval_policy(pol, seed=seed * 1000 + 123)
    train_ppo_maison(pol, ref, value, RM_SEED_LOCAL, seed=seed)
    out['maison'] = dict(r=eval_policy(pol, seed=seed * 1000 + 123), t=time.time() - t0, r0=r0)

    t0 = time.time()
    tr = train_grpo_trl(seed)
    out['trl'] = dict(r=eval_policy(tr.model, seed=seed * 1000 + 123), t=time.time() - t0, r0=r0)
    return out

RES = {}
for s in SEEDS[:1]:
    RES[s] = run_seed(s)
    m, c = RES[s]['maison'], RES[s]['trl']
    print(f"seed {s}: maison {m['r0']:.3f}->{m['r']:.3f} ({m['t']:.0f}s) | "
          f"trl {c['r0']:.3f}->{c['r']:.3f} ({c['t']:.0f}s)")
print("(les seeds restantes tournent -- cellule suivante)")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/28 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/28 [00:00<?, ?it/s]

{'loss': '-0.3013', 'grad_norm': '1.19', 'learning_rate': '9.028e-05', 'num_tokens': '1990', 'completions/mean_length': '6.773', 'completions/min_length': '1', 'completions/max_length': '9', 'completions/clipped_ratio': '0.4375', 'completions/mean_terminated_length': '5.061', 'completions/min_terminated_length': '1', 'completions/max_terminated_length': '9', 'rewards/reward_rm/mean': '-1.828', 'rewards/reward_rm/std': '2.583', 'reward': '-1.828', 'reward_std': '2.583', 'frac_reward_zero_std': '0', 'kl': '0.008674', 'entropy': '2.369', 'clip_ratio/low_mean': '0', 'clip_ratio/high_mean': '0', 'clip_ratio/region_mean': '0', 'clip_ratio/low_min': '0', 'clip_ratio/high_max': '0', 'step_time': '0.3858', 'epoch': '0.3333'}


{'loss': '-0.2663', 'grad_norm': '0.8682', 'learning_rate': '7.917e-05', 'num_tokens': '4079', 'completions/mean_length': '7.16', 'completions/min_length': '1.125', 'completions/max_length': '9', 'completions/clipped_ratio': '0.5', 'completions/mean_terminated_length': '5.343', 'completions/min_terminated_length': '1.125', 'completions/max_terminated_length': '9', 'rewards/reward_rm/mean': '-1.225', 'rewards/reward_rm/std': '2.281', 'reward': '-1.225', 'reward_std': '2.281', 'frac_reward_zero_std': '0', 'kl': '0.02859', 'entropy': '2.358', 'clip_ratio/low_mean': '0', 'clip_ratio/high_mean': '0', 'clip_ratio/region_mean': '0', 'clip_ratio/low_min': '0', 'clip_ratio/high_max': '0', 'step_time': '0.3677', 'epoch': '0.6667'}


...

{'loss': '-0.2078', 'grad_norm': '0.9393', 'learning_rate': '3.472e-05', 'num_tokens': '1.305e+04', 'completions/mean_length': '7.691', 'completions/min_length': '1.25', 'completions/max_length': '9', 'completions/clipped_ratio': '0.6172', 'completions/mean_terminated_length': '5.497', 'completions/min_terminated_length': '1.25', 'completions/max_terminated_length': '9', 'rewards/reward_rm/mean': '-0.2605', 'rewards/reward_rm/std': '2.036', 'reward': '-0.2605', 'reward_std': '2.036', 'frac_reward_zero_std': '0', 'kl': '0.08299', 'entropy': '2.322', 'clip_ratio/low_mean': '0', 'clip_ratio/high_mean': '0', 'clip_ratio/region_mean': '0', 'clip_ratio/low_min': '0', 'clip_ratio/high_max': '0', 'step_time': '0.3637', 'epoch': '2'}


{'loss': '-0.1951', 'grad_norm': '0.7103', 'learning_rate': '2.361e-05', 'num_tokens': '1.528e+04', 'completions/mean_length': '7.703', 'completions/min_length': '1.5', 'completions/max_length': '9', 'completions/clipped_ratio': '0.6445', 'completions/mean_terminated_length': '5.33', 'completions/min_terminated_length': '1.5', 'completions/max_terminated_length': '9', 'rewards/reward_rm/mean': '-0.4443', 'rewards/reward_rm/std': '2.101', 'reward': '-0.4443', 'reward_std': '2.101', 'frac_reward_zero_std': '0', 'kl': '0.08399', 'entropy': '2.317', 'clip_ratio/low_mean': '0', 'clip_ratio/high_mean': '0', 'clip_ratio/region_mean': '0', 'clip_ratio/low_min': '0', 'clip_ratio/high_max': '0', 'step_time': '0.3536', 'epoch': '2.333'}


{'loss': '-0.1918', 'grad_norm': '0.6812', 'learning_rate': '1.25e-05', 'num_tokens': '1.753e+04', 'completions/mean_length': '7.809', 'completions/min_length': '1.125', 'completions/max_length': '9', 'completions/clipped_ratio': '0.6797', 'completions/mean_terminated_length': '5.298', 'completions/min_terminated_length': '1.125', 'completions/max_terminated_length': '9', 'rewards/reward_rm/mean': '-0.4361', 'rewards/reward_rm/std': '2.122', 'reward': '-0.4361', 'reward_std': '2.122', 'frac_reward_zero_std': '0', 'kl': '0.08295', 'entropy': '2.313', 'clip_ratio/low_mean': '0', 'clip_ratio/high_mean': '0', 'clip_ratio/region_mean': '0', 'clip_ratio/low_min': '0', 'clip_ratio/high_max': '0', 'step_time': '0.3482', 'epoch': '2.667'}


{'loss': '-0.1609', 'grad_norm': '0.6709', 'learning_rate': '1.389e-06', 'num_tokens': '1.984e+04', 'completions/mean_length': '8.004', 'completions/min_length': '1.625', 'completions/max_length': '9', 'completions/clipped_ratio': '0.6953', 'completions/mean_terminated_length': '5.711', 'completions/min_terminated_length': '1.625', 'completions/max_terminated_length': '9', 'rewards/reward_rm/mean': '-0.2654', 'rewards/reward_rm/std': '1.852', 'reward': '-0.2654', 'reward_std': '1.852', 'frac_reward_zero_std': '0', 'kl': '0.08312', 'entropy': '2.311', 'clip_ratio/low_mean': '0', 'clip_ratio/high_mean': '0', 'clip_ratio/region_mean': '0', 'clip_ratio/low_min': '0', 'clip_ratio/high_max': '0', 'step_time': '0.3311', 'epoch': '3'}
{'train_runtime': '26.22', 'train_samples_per_second': '10.98', 'train_steps_per_second': '2.746', 'train_loss': '-0.2145', 'epoch': '3'}


seed 0: maison 0.219->0.815 (66s) | trl 0.219->0.880 (43s)
(les seeds restantes tournent -- cellule suivante)


In [10]:
for s in SEEDS[1:]:
    RES[s] = run_seed(s)
    m, c = RES[s]['maison'], RES[s]['trl']
    print(f"seed {s}: maison {m['r0']:.3f}->{m['r']:.3f} ({m['t']:.0f}s) | "
          f"trl {c['r0']:.3f}->{c['r']:.3f} ({c['t']:.0f}s)")

gains_m = np.array([RES[s]['maison']['r'] - RES[s]['maison']['r0'] for s in SEEDS])
gains_c = np.array([RES[s]['trl']['r'] - RES[s]['trl']['r0'] for s in SEEDS])
ts_m = np.array([RES[s]['maison']['t'] for s in SEEDS])
ts_c = np.array([RES[s]['trl']['t'] for s in SEEDS])
print(f"\ngain maison : {gains_m.mean():+.3f} ± {gains_m.std():.3f}  (temps moyen {ts_m.mean():.0f}s)")
print(f"gain trl    : {gains_c.mean():+.3f} ± {gains_c.std():.3f}  (temps moyen {ts_c.mean():.0f}s)")
diff = gains_c.mean() - gains_m.mean()
sig = max(gains_m.std(), gains_c.std())
print(f"écart trl-maison : {diff:+.3f} (2σ = {2*sig:.3f}) -> "
      + ("INCONCLUSIVE" if abs(diff) < 2 * sig else f"{'trl' if diff > 0 else 'maison'} devant"))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/28 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/28 [00:00<?, ?it/s]

{'loss': '-0.3009', 'grad_norm': '1', 'learning_rate': '9.028e-05', 'num_tokens': '2034', 'completions/mean_length': '6.945', 'completions/min_length': '1', 'completions/max_length': '9', 'completions/clipped_ratio': '0.4883', 'completions/mean_terminated_length': '5.067', 'completions/min_terminated_length': '1', 'completions/max_terminated_length': '9', 'rewards/reward_rm/mean': '-1.404', 'rewards/reward_rm/std': '1.718', 'reward': '-1.404', 'reward_std': '1.718', 'frac_reward_zero_std': '0', 'kl': '0.005393', 'entropy': '2.375', 'clip_ratio/low_mean': '0', 'clip_ratio/high_mean': '0', 'clip_ratio/region_mean': '0', 'clip_ratio/low_min': '0', 'clip_ratio/high_max': '0', 'step_time': '0.3588', 'epoch': '0.3333'}


{'loss': '-0.2262', 'grad_norm': '1.041', 'learning_rate': '7.917e-05', 'num_tokens': '4150', 'completions/mean_length': '7.266', 'completions/min_length': '1.125', 'completions/max_length': '9', 'completions/clipped_ratio': '0.5117', 'completions/mean_terminated_length': '5.43', 'completions/min_terminated_length': '1.125', 'completions/max_terminated_length': '9', 'rewards/reward_rm/mean': '-1.081', 'rewards/reward_rm/std': '1.593', 'reward': '-1.081', 'reward_std': '1.593', 'frac_reward_zero_std': '0', 'kl': '0.01856', 'entropy': '2.368', 'clip_ratio/low_mean': '0', 'clip_ratio/high_mean': '0', 'clip_ratio/region_mean': '0', 'clip_ratio/low_min': '0', 'clip_ratio/high_max': '0', 'step_time': '0.3527', 'epoch': '0.6667'}


...

{'loss': '-0.229', 'grad_norm': '0.8397', 'learning_rate': '3.472e-05', 'num_tokens': '1.276e+04', 'completions/mean_length': '7.562', 'completions/min_length': '1.25', 'completions/max_length': '9', 'completions/clipped_ratio': '0.6484', 'completions/mean_terminated_length': '5.051', 'completions/min_terminated_length': '1.25', 'completions/max_terminated_length': '9', 'rewards/reward_rm/mean': '-0.5854', 'rewards/reward_rm/std': '0.9832', 'reward': '-0.5854', 'reward_std': '0.9832', 'frac_reward_zero_std': '0', 'kl': '0.09683', 'entropy': '2.29', 'clip_ratio/low_mean': '0', 'clip_ratio/high_mean': '0', 'clip_ratio/region_mean': '0', 'clip_ratio/low_min': '0', 'clip_ratio/high_max': '0', 'step_time': '0.3801', 'epoch': '2'}


{'loss': '-0.1912', 'grad_norm': '0.8303', 'learning_rate': '2.361e-05', 'num_tokens': '1.499e+04', 'completions/mean_length': '7.723', 'completions/min_length': '1.25', 'completions/max_length': '9', 'completions/clipped_ratio': '0.6289', 'completions/mean_terminated_length': '5.601', 'completions/min_terminated_length': '1.25', 'completions/max_terminated_length': '8.875', 'rewards/reward_rm/mean': '-0.5513', 'rewards/reward_rm/std': '0.9701', 'reward': '-0.5513', 'reward_std': '0.9701', 'frac_reward_zero_std': '0', 'kl': '0.1063', 'entropy': '2.283', 'clip_ratio/low_mean': '0', 'clip_ratio/high_mean': '0', 'clip_ratio/region_mean': '0', 'clip_ratio/low_min': '0', 'clip_ratio/high_max': '0', 'step_time': '0.3381', 'epoch': '2.333'}


{'loss': '-0.1821', 'grad_norm': '0.8611', 'learning_rate': '1.25e-05', 'num_tokens': '1.724e+04', 'completions/mean_length': '7.793', 'completions/min_length': '1.5', 'completions/max_length': '9', 'completions/clipped_ratio': '0.6484', 'completions/mean_terminated_length': '5.622', 'completions/min_terminated_length': '1.5', 'completions/max_terminated_length': '9', 'rewards/reward_rm/mean': '-0.5003', 'rewards/reward_rm/std': '0.9165', 'reward': '-0.5003', 'reward_std': '0.9165', 'frac_reward_zero_std': '0', 'kl': '0.1089', 'entropy': '2.277', 'clip_ratio/low_mean': '0', 'clip_ratio/high_mean': '0', 'clip_ratio/region_mean': '0', 'clip_ratio/low_min': '0', 'clip_ratio/high_max': '0', 'step_time': '0.3419', 'epoch': '2.667'}


{'loss': '-0.1746', 'grad_norm': '0.6092', 'learning_rate': '1.389e-06', 'num_tokens': '1.951e+04', 'completions/mean_length': '7.855', 'completions/min_length': '1.75', 'completions/max_length': '9', 'completions/clipped_ratio': '0.6523', 'completions/mean_terminated_length': '5.873', 'completions/min_terminated_length': '1.75', 'completions/max_terminated_length': '9', 'rewards/reward_rm/mean': '-0.4444', 'rewards/reward_rm/std': '0.9282', 'reward': '-0.4444', 'reward_std': '0.9282', 'frac_reward_zero_std': '0', 'kl': '0.1152', 'entropy': '2.274', 'clip_ratio/low_mean': '0', 'clip_ratio/high_mean': '0', 'clip_ratio/region_mean': '0', 'clip_ratio/low_min': '0', 'clip_ratio/high_max': '0', 'step_time': '0.3295', 'epoch': '3'}
{'train_runtime': '26.26', 'train_samples_per_second': '10.97', 'train_steps_per_second': '2.742', 'train_loss': '-0.2278', 'epoch': '3'}


seed 42: maison 0.341->0.949 (56s) | trl 0.341->1.089 (41s)

gain maison : +0.588 ± 0.070  (temps moyen 61s)
gain trl    : +0.551 ± 0.183  (temps moyen 42s)
écart trl-maison : -0.037 (2σ = 0.366) -> INCONCLUSIVE


## 7. From scratch vs SOTA : le tableau

Lignes de code comptées sur les sources du notebook livré (lignes non vides des
définitions, docstrings incluses) : bras maison = ValueNet + token_logps + rollout +
seqs_to_x + train_ppo_maison = 60 ; bras trl = reward_rm + train_grpo_trl (incluant
le bloc GRPOConfig et la construction du dataset) = 36.

In [11]:
# LOC pré-mesurées sur les sources du notebook livré (lignes non vides, docstrings incluses)
loc_m = 60
loc_c = 36

print(f"{'':<24}{'PPO maison':>20}{'trl.GRPOTrainer':>20}")
print(f"{'gain vraie récomp.':<24}{gains_m.mean():>20.3f}{gains_c.mean():>20.3f}")
print(f"{'dispersion (std)':<24}{gains_m.std():>20.3f}{gains_c.std():>20.3f}")
print(f"{'temps moyen / run':<24}{ts_m.mean():>19.0f}s{ts_c.mean():>19.0f}s")
print(f"{'lignes de code':<24}{loc_m:>20}{loc_c:>20}")
print(f"{'critic':<24}{'value net (appris)':>20}{'groupe (structural)':>20}")
print(f"{'dépendances':<24}{'torch':>20}{'+trl, datasets':>20}")

                                  PPO maison     trl.GRPOTrainer
gain vraie récomp.                     0.588               0.551
dispersion (std)                       0.070               0.183
temps moyen / run                        61s                 42s
lignes de code                            60                  36
critic                    value net (appris) groupe (structural)
dépendances                            torch      +trl, datasets


**Pourquoi du from scratch.** La boucle PPO maison rend visibles les trois pièces que
GRPO factorise : le critic (vous l'entraînez, vous voyez sa `vloss` dériver quand la
politique bouge), le clip (vous pouvez le désactiver et regarder exploser), la
pénalité KL (même $\beta$ des deux côtés — comparez les colonnes `kl`). Quand le bras
trl et le bras maison arrivent au même niveau, ce n'est pas une coïncidence : c'est
le même théorème de politique améliorée, deux estimateurs de l'avantage.

**Quand le SOTA.** `GRPOTrainer` n'existe pas pour battre une boucle de 60 lignes sur
un monde à 11 tokens — il existe parce que la même interface accepte demain Qwen à 7B,
vLLM, LoRA, la distribution multi-GPU, et surtout **des rewards vérifiables** : la
signature `reward_funcs(prompts, completions, completion_ids)` a fait de GRPO le
moteur du RLVR. Le from scratch apprend **ce que fait l'option** ; le SOTA est **le
moyen de ne pas l'écrire soi-même** en production.

**Et le PPO dans tout ça ?** Il n'a pas disparu du champ théorique — c'est toujours
le socle (GRPO = PPO sans critic, baseline déplacée dans le groupe). Il a disparu du
**catalogue trl**, et c'est l'enseignement transversal de ce notebook : une
comparaison « SOTA » est datée par construction. `rlpt_1` a appris PPO ; `rlpt_0e`
a appris DPO ; ici le SOTA en ligne s'appelle GRPO. La pile bouge, la mathématique
de la baseline reste.

## 8. Ce qu'il faut retenir

| Leçon | Mesure dans ce notebook |
|---|---|
| La baseline est LE choix de design | value net appris (PPO) vs moyenne de groupe (GRPO) — même récompense terminale, même budget |
| RLHF optimise le juge, pas le vrai | le gain se mesure en **vraie** récompense, jamais en score RM |
| GRPO = PPO sans critic | mêmes gains à budget égal (section 6), zéro paramètre de value |
| `trl.PPOTrainer` a existé | supprimé upstream — une comparaison SOTA est **datée**, le notebook documente la succession |
| L'interface reward est le produit | `reward_funcs(prompts, completions, completion_ids)` : le juge est une fonction, d'où le RLVR |

**Ce que ce notebook ajoute à la série** : `rlpt_1` écrivait PPO from scratch sur sa
propre tâche ; `rlpt_0e` branchait le SOTA **offline** (DPO) sur le monde de `rlpt_0`.
Ici la boucle fermée **juge → échantillonnage → politique** passe en ligne, le SOTA
de l'époque (`GRPOTrainer`) est départagé contre la recette maison à budget exact de
rollouts, et la disparition de `PPOTrainer` — le sujet initial de l'issue — est
mesurée et documentée.

## Exercices

Trois exercices, du plus guidé au plus ouvert. Ils étendent les fonctions définies
ci-dessus — aucune nouvelle dépendance.

In [12]:
# Exercice 1 -- Le coefficient KL : balayer beta
# TODO etudiant
# Etape 1 : re-entraîner le bras maison pour beta dans [0.0, 0.01, 0.05, 0.2]
#            (train_ppo_maison(pol, ref, value, RM, seed=0, beta=...))
# Etape 2 : pour chaque beta, mesurer (a) le gain en vraie récompense, (b) la KL finale
#            contre ref -- la KL est deja dans hist (dernier element : hist[-1]['kl'])
# Etape 3 : tracer le compromis gain vs KL. Ou se situe le "coude" ?
resultats_beta = None  # TODO etudiant
print("Exercice a completer")

Exercice a completer


In [13]:
# Exercice 2 -- RLOO : la baseline leave-one-out, a la main
# TODO etudiant
# Etape 1 : copier rollout/seqs_to_x ; generer G=8 reponses par prompt (meme prompt G fois)
# Etape 2 : avantage RLOO de la reponse i : A_i = r_i - mean(r_j, j != i)
#            (la baseline de i ignore i -- c'est tout le "leave-one-out")
# Etape 3 : re-utiliser la boucle PPO (clip + KL) avec ce nouvel avantage, sans value net
# Etape 4 : comparer a PPO et a GRPO sur la meme metrique -- RLOO est le 3e estimateur
resultats_rloo = None  # TODO etudiant
print("Exercice a completer")

Exercice a completer


In [14]:
# Exercice 3 -- Reward hacking : un juge biaise par la longueur
# TODO etudiant
# Etape 1 : construire un RM biaise : re-entraîner RewardModel sur des paires dont le
#            gagnant est artificiellement allonge (label flippe si len(gagnant) > len(perdant))
# Etape 2 : entraîner la politique maison contre CE juge (meme boucle, rm=biasé)
# Etape 3 : mesurer l'ecart : score du juge biaise vs VRAIE récompense de la politique
#            -- la politique exploite-t-elle le biais ? (c'est rlpt_3, mais en ligne)
resultats_hack = None  # TODO etudiant
print("Exercice a completer")

Exercice a completer


## Références

- `rlpt_0` — le monde synthétique et le reward model Bradley-Terry (même dépôt).
- `rlpt_1` — PPO from scratch pour l'alignement d'un LM (même dépôt).
- `rlpt_0e` — le geste SOTA offline : `trl.DPOTrainer` sur le même monde (même dépôt).
- `rlpt_2` / `PT-11a` — GRPO respectivement à la main sur Qwen et à l'échelle avec
  `trl.GRPOTrainer` + rewards vérifiables (même dépôt).
- Shao et al. 2024, *DeepSeekMath* — la section qui introduit GRPO (group relative
  policy optimization).
- Schulman et al. 2017, *Proximal Policy Optimization* — le clip et le critic que
  GRPO supprime.
- Rafailov et al. 2023, *Direct Preference Optimization* — la voie offline de `rlpt_0e`.